# Multi-Factor Models
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Estimate factor loadings** of a portfolio against FF3, FF5, FF6
2. **Interpret factor loadings** — what does β=0.5 on SMB mean?
3. **Apply the "factor zoo" warning** — most published factors don't replicate
4. **Choose the right factor model** for the question you're asking
5. **Audit AI-generated multi-factor analyses** — sample windows, factor data versions

## 📋 TOC
1. [Setup](#setup)  2. [From One Factor to Many](#many)
3. [Pitfall Checklist](#pitfalls)  4. [The Standard Models](#standard)
5. [The Factor Zoo](#zoo)  6. [Choosing the Right Model](#choose)
7. [🎯 Challenge: Decompose a Fund](#challenge)
8. [Submission](#submit)  9. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## From One Factor to Many <a id="many"></a>

Single-factor CAPM:

$$r_t^e = \alpha + \beta \cdot MKT_t + \epsilon_t$$

Multi-factor:

$$r_t^e = \alpha + \sum_{k=1}^K \beta_k \cdot F_{k,t} + \epsilon_t$$

**Why add factors?** Because empirically, the market alone leaves systematic
patterns in returns. Small stocks earn more than large. Value stocks earn
more than growth. Recent winners earn more than recent losers. Multi-factor
models capture these patterns explicitly.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Wrong factor data version** | Ken French updates factors periodically; old data has slightly different SMB | Pull factors fresh from the source |
| 2 | **Daily vs monthly inconsistency** | Mixing daily fund returns with monthly factors | Use the same frequency throughout |
| 3 | **Time-zone / day-end mismatches** | Some factors use 4pm ET close; some use closing auction | Compute corr between MKT and SPY excess; should be ~0.99 |
| 4 | **Sample period changes everything** | A fund's HML loading may be +0.3 in one decade and -0.3 in another | Report rolling loadings, not just point estimates |
| 5 | **Confusing factor "tilt" with skill** | A fund with high HML loading earned the value premium, not alpha | If alpha disappears when you add HML, the "skill" was just exposure |

---
## The Standard Models <a id="standard"></a>

| Model | Factors | What they capture |
|-------|---------|-------------------|
| **CAPM** | MKT | Market exposure |
| **FF3** | MKT, SMB, HML | + Size, value |
| **FF5** | + RMW, CMA | + Profitability, investment |
| **FF6** | + MOM | + Momentum |
| **Q-Factor** | MKT, ME, I/A, ROE | Hou-Xue-Zhang alternative |
| **AQR 7-factor** | MKT, SMB, HML, MOM, BAB, QMJ, ... | Adds proprietary factors |

For **academic work**, FF6 is the default. For **practitioner work**, AQR's
or BlackRock's proprietary multi-factor models are common. The right model
depends on what's commercially available and what you're trying to explain.

---
## The Factor Zoo <a id="zoo"></a>

Harvey, Liu & Zhu (2016) document **316+ "factors"** published in academic
journals. Their estimate: **most won't replicate.**

The reasons:
- **Multiple testing.** If you try 1000 random characteristics, ~50 will be
  "significant" by chance with no economic content.
- **Publication bias.** Negative results don't get published.
- **Data mining.** Researchers try variants until something works.

**Hou, Xue & Zhang (2020)** replicated 452 factors with stricter methodology;
~65% failed to clear the t-stat threshold.

> **🤖 AI-Era Insight**
>
> An LLM can readily list dozens of factor models. It can't tell you which
> ones have survived honest replication. **Be skeptical of any "new factor."**

---
## Choosing the Right Model <a id="choose"></a>

| If you're trying to ... | Use ... |
|-------------------------|---------|
| Evaluate a long-only equity fund | FF5 or FF6 |
| Evaluate a market-neutral hedge fund | FF6 + BAB; consider AQR 7-factor |
| Risk-manage a portfolio | Whatever covariance model is operational |
| Test a new "anomaly" | Both FF5 and HXZ q-factor to be defensive |
| Allocate across asset classes | Different model entirely (macro factors) |

---
## 🎯 Challenge: Decompose a Fund <a id="challenge"></a>

> **Setup.** A fund has the following loadings from a 10-year FF5 regression:
> - α = 1.5%/yr (t = 1.2)
> - β_MKT = 1.10 (t = 22)
> - β_SMB = 0.30 (t = 5.5)
> - β_HML = -0.20 (t = -3.2)
> - β_RMW = 0.15 (t = 2.8)
> - β_CMA = -0.10 (t = -1.6)
>
> Average factor premia over the same sample (annualized): MKT=8%, SMB=2%,
> HML=3%, RMW=4%, CMA=2%.

### Q1 — Decompose expected return into factor contributions

Each factor's contribution = β × factor premium.

> **📌 Required:**
> ```python
> contrib_MKT   = ____   # 1.10 * 0.08
> contrib_SMB   = ____
> contrib_HML   = ____
> contrib_RMW   = ____
> contrib_CMA   = ____
> ```

In [ ]:
b_mkt, b_smb, b_hml, b_rmw, b_cma = 1.10, 0.30, -0.20, 0.15, -0.10
p_mkt, p_smb, p_hml, p_rmw, p_cma = 0.08, 0.02, 0.03, 0.04, 0.02

contrib_MKT = ____
contrib_SMB = ____
contrib_HML = ____
contrib_RMW = ____
contrib_CMA = ____

total_factor_return = (contrib_MKT + contrib_SMB + contrib_HML + contrib_RMW + contrib_CMA)
print(f"MKT contribution:  {contrib_MKT:+.2%}")
print(f"SMB contribution:  {contrib_SMB:+.2%}")
print(f"HML contribution:  {contrib_HML:+.2%}")
print(f"RMW contribution:  {contrib_RMW:+.2%}")
print(f"CMA contribution:  {contrib_CMA:+.2%}")
print(f"Total from factors: {total_factor_return:+.2%}")

### Q2 — Is the alpha real?

The alpha is 1.5%/yr with t-stat 1.2. Is that statistically distinguishable from zero?

> **📌 Required:**
> ```python
> alpha_is_significant = ____   # 1.0 if |t| > 2, 0.0 otherwise
> ```

In [ ]:
alpha_t = 1.2

alpha_is_significant = ____
print(f"Alpha statistically significant? {bool(alpha_is_significant)}")

### Q3 — Total return decomposition

If the fund's total annual return was 12%, how much of it came from factor exposure vs alpha?

> **📌 Required:**
> ```python
> fraction_from_factors = ____   # total_factor_return / 0.12
> ```

In [ ]:
total_return = 0.12

fraction_from_factors = ____
print(f"Fraction from factors: {fraction_from_factors:.1%}")

### Q4 — Memo

Max 5 sentences. Recommend whether to invest. Cite (i) which factor exposures
drive returns, (ii) whether alpha is significant, (iii) what the fund manager
should be paid for vs what's commodity beta.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["contrib_MKT", "contrib_SMB", "contrib_HML", "contrib_RMW", "contrib_CMA", "alpha_is_significant", "fraction_from_factors", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "MultiFactorModels_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **FF6 is the academic default.** Adds size, value, profitability, investment, momentum to CAPM.
2. **Alpha shrinks as you add factors.** Most "alpha" is re-classified factor exposure.
3. **The factor zoo is mostly noise.** Be skeptical of new factors that haven't been independently replicated.
4. **Decomposition reveals what the manager actually does.** Often it's commodity factor exposure, not skill.
5. **AI runs the regression. You pick the right factors and interpret what's left.**